# Explore the local lake

Run `docker compose run --rm platform` first to build `.local-lake`, then start JupyterLab with `docker compose --profile notebooks up jupyter`.

In [ ]:
from pyspark.sql import functions as F

from zingyestates.config import PlatformConfig
from zingyestates.spark import get_spark

spark = get_spark("explore")
cfg = PlatformConfig.for_local("/workspace/.local-lake")


def gold(table):
    return spark.read.format("delta").load(cfg.table_path("gold", table))

## Market summary by city

In [ ]:
gold("agg_market_monthly").groupBy("city", "state").agg(
    F.sum("sales_count").alias("sales"),
    F.sum("total_volume").alias("volume"),
    F.avg("avg_price_per_sqft").alias("avg_price_per_sqft"),
).orderBy(F.desc("volume")).toPandas()

## Top agents by closed sales volume

In [ ]:
(gold("fact_sales").join(gold("dim_agent"), "agent_id")
 .groupBy("full_name", "office").agg(F.count("*").alias("deals"), F.sum("sale_price").alias("volume"))
 .orderBy(F.desc("volume")).limit(10).toPandas())

## Quarantined records and the rules they failed

In [ ]:
for entity in ["properties", "agents", "sales_transactions", "leases", "listings"]:
    q = spark.read.format("delta").load(cfg.table_path("quarantine", entity))
    print(entity)
    q.select("_dq_failures", *q.columns[:3]).show(truncate=False)

## Delta time travel: silver history

In [ ]:
history = spark.sql(f"DESCRIBE HISTORY delta.`{cfg.table_path('silver', 'sales_transactions')}`")
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)